# 📊 Credit Risk - Interactive EDA with Plotly
This notebook provides an interactive deep dive into the credit risk dataset using high-quality Plotly visualizations.

In [16]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

# Setting a clean template
pio.templates.default = 'plotly_white'

df = pd.read_csv('../data/processed/credit_risk_cleaned.csv')
df.head()

,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0,0.10,N,2
1,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1,0.57,N,3
2,23,65500,RENT,4.0,MEDICAL,C,35000,15.23,1,0.53,N,2
3,24,54400,RENT,8.0,MEDICAL,C,35000,14.27,1,0.55,Y,4
4,21,9900,OWN,2.0,VENTURE,A,2500,7.14,1,0.25,N,2


## 1. Loan Status Distribution
Interactively exploring the target variable imbalance.

In [17]:
# Interactive Count Plot
fig = px.histogram(df, x='loan_status', color='loan_status', 
                   title='Distribution of Loan Status',
                   labels={'loan_status':'Default Status'},
                   color_discrete_map={0: '#4C72B0', 1: '#C44E52'},
                   text_auto=True)
fig.update_layout(bargap=0.2, showlegend=False)
fig.show()

## 2. Advanced Global Relationships
A subset of numerical features shown in an interactive scatter matrix.

In [18]:
# Interactive Scatter Matrix (subset of columns for performance)
fig = px.scatter_matrix(df, dimensions=['person_age', 'person_income', 'loan_amnt', 'loan_int_rate'],
                        color='loan_status', opacity=0.5,
                        title='Scatter Matrix of Key Financial Features')
fig.update_layout(height=800)
fig.show()

## 3. Data Quality Analysis
Visualizing missing values and checking distributions.

In [19]:
# Missing Values Heatmap
missing_data = df.isnull().astype(int)
fig = px.imshow(missing_data, 
                labels=dict(x='Features', y='Index', color='Is Missing'),
                title='Missingness Heatmap',
                color_continuous_scale='Viridis')
fig.show()

print('\nMissing values count:')
print(df.isnull().sum())


Missing values count:
person_age                    0
person_income                 0
person_home_ownership         0
person_emp_length             0
loan_intent                   0
loan_grade                    0
loan_amnt                     0
loan_int_rate                 0
loan_status                   0
loan_percent_income           0
cb_person_default_on_file     0
cb_person_cred_hist_length    0
dtype: int64


## 4. Feature Distribution & Outliers
Interactive boxplots allow for hovering over specific outlier values.

In [20]:
# Interactive Multi-Box Plot
features_to_check = ['person_age', 'person_income', 'person_emp_length', 'loan_amnt']
fig = go.Figure()

for feature in features_to_check:
    fig.add_trace(go.Box(y=df[feature], name=feature))

fig.update_layout(title='Interactive Boxplots for Outlier Detection', height=600)
fig.show()

## 5. Bivariate Analysis: Features vs Target
Detailed interactive analysis showing the breakdown of defaults across different categories.

In [21]:
# Interactive Split Analysis
# 1. Income vs Loan Status
fig1 = px.box(df, x='loan_status', y='person_income', color='loan_status',
              title='Income vs Loan Status', points='outliers')
fig1.show()

# 2. Loan Amount vs Loan Status
fig2 = px.box(df, x='loan_status', y='loan_amnt', color='loan_status',
              title='Loan Amount vs Loan Status')
fig2.show()

# 3. Loan Grade vs Loan Status (Normalized Percentage)
fig3 = px.histogram(df, x='loan_grade', color='loan_status',
                    barnorm='percent', title='Default Percentage per Loan Grade',
                    category_orders={'loan_grade': sorted(df['loan_grade'].unique())})
fig3.show()

## 6. Correlation Analysis
An interactive heatmap with tooltips for exact correlation values.

### 🔍 Understanding the Correlation Heatmap
The heatmap uses the **Pearson Correlation Coefficient** (r) to show how numerical variables relate to each other:

- **Magnitude (|r|)**: The closer to **1** or **-1**, the stronger the relationship. **0** means no linear relationship.
- **Positive Correlation (0 to 1)**: Variables move in the same direction. (e.g., Higher Income → Higher Loan Amount).
- **Negative Correlation (-1 to 0)**: Variables move in opposite directions. (e.g., Higher Loan Grade → Lower Default Rate).

> [!IMPORTANT]
> - **Strong Predictors**: Look for features with high correlation to `loan_status`.
> - **Multicollinearity**: If two features have a correlation > 0.8, they might be redundant.


In [22]:
# Interactive Correlation Heatmap
numerical_df = df.select_dtypes(include=['float64', 'int64'])
corr_matrix = numerical_df.corr()

fig = px.imshow(corr_matrix, text_auto='.2f', 
                title='Interactive Correlation Heatmap',
                color_continuous_scale='RdBu_r', range_color=[-1, 1])
fig.update_layout(width=800, height=800)
fig.show()

## 7. Refined Income Analysis (Scatter with Marginals)
To better understand `person_income` and its outliers, we use scatter plots with marginal boxplots. This provides a clear view of how income relates to other features like the loan amount.

In [23]:
# 1. Income vs Loan Amount with Marginal Distribution Plots
fig1 = px.scatter(df, x='person_income', y='loan_amnt', color='loan_status',
                   marginal_x='box', marginal_y='violin',
                   title='Income vs Loan Amount (Marginal Boxes)',
                   labels={'person_income': 'Person Income', 'loan_amnt': 'Loan Amount'},
                   opacity=0.6)
fig1.update_layout(height=700)
fig1.show()

# 2. Income vs Age with Marginal Distributions
# This helps visualize if outliers are unique to a certain age group
fig2 = px.scatter(df, x='person_age', y='person_income', color='loan_status',
                   marginal_x='histogram', marginal_y='box',
                   title='Age vs Income (Marginal Boxes)',
                   labels={'person_age': 'Age', 'person_income': 'Income'})
fig2.update_layout(height=700)
fig2.show()

> [!TIP]
> In Plotly, if the income outliers make the plot hard to read, double-click the axis or use the 'Box Select' tool to zoom in on a specific income bracket interactively.